# SUBIR_RESULTADOS_GITHUB — Publicación de resultados (Cuenca Amaime)

**Propósito único de este notebook: publicar en GitHub los resultados que
YA EXISTEN en `RESULTADOS/`**, generados por `RUSLE_PIPELINE_COMPLETO.ipynb`.

Este notebook **NO hace ninguna de estas cosas**:
- NO calcula ni recalcula RUSLE, LS, Susceptibilidad ni CHIRPS.
- NO descarga nada de Google Earth Engine.
- NO modifica el Excel original ni ningún dato de entrada.
- NO reprocesa, reproyecta ni reescribe ningún ráster.
- NO mezcla resultados de distintas categorías.
- NO crea un pipeline alterno.

Si `RESULTADOS/` no existe o está incompleto, **ejecute primero
`RUSLE_PIPELINE_COMPLETO.ipynb`** — este notebook se detiene si no
encuentra resultados ya generados.

**Seguridad**: el paso final (`git push`) requiere que usted cambie
manualmente `CONFIRMAR_PUSH = True` en la celda correspondiente. Mientras
esa bandera esté en `False`, el notebook prepara todo (`git add`,
`git commit`) pero NO publica nada al repositorio remoto.


In [ ]:

import subprocess
from pathlib import Path
from datetime import datetime

RAIZ = Path(r"D:/Diego Angrino Chiran/Documentos/Cenicaña/SOLIX/RUSLE_Amaime")
DIR_RESULTADOS = RAIZ / "RESULTADOS"
NOTEBOOK_PIPELINE = RAIZ / "Script" / "RUSLE_PIPELINE_COMPLETO.ipynb"
NOTEBOOK_GITHUB = RAIZ / "Script" / "SUBIR_RESULTADOS_GITHUB.ipynb"

assert DIR_RESULTADOS.exists(), (
    f"No existe {DIR_RESULTADOS}. Ejecute primero RUSLE_PIPELINE_COMPLETO.ipynb "
    "para generar los resultados antes de publicarlos."
)

def run(cmd, check=True):
    r = subprocess.run(cmd, cwd=str(RAIZ), capture_output=True, text=True, shell=False)
    print(f"$ {' '.join(cmd)}")
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.stderr.strip():
        print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f"Comando fallo (codigo {r.returncode}): {' '.join(cmd)}")
    return r

print(f"Raiz del proyecto: {RAIZ}")
print(f"Resultados a publicar: {DIR_RESULTADOS}")


## 1) Inventario de resultados ya generados (respetando categorías)

Solo se lista lo que ya existe — no se genera ni modifica nada aquí.


In [ ]:

CATEGORIAS_ESPERADAS = [
    "01_FACTORES_RUSLE", "02_PERDIDA_SUELO_RUSLE", "03_SUSCEPTIBILIDAD",
    "04_CHIRPS", "05_BASE_DATOS_PROCESADA", "06_VALIDACION",
]
print("Inventario de RESULTADOS/ por categoria (solo lectura):\n")
total_archivos, total_bytes = 0, 0
for cat in CATEGORIAS_ESPERADAS:
    dir_cat = DIR_RESULTADOS / cat
    if not dir_cat.exists():
        print(f"  [AUSENTE] {cat}")
        continue
    archivos = [p for p in dir_cat.rglob("*") if p.is_file()]
    tam = sum(p.stat().st_size for p in archivos)
    total_archivos += len(archivos)
    total_bytes += tam
    print(f"  {cat}: {len(archivos)} archivos, {tam/1e6:.1f} MB")

print(f"\nTOTAL: {total_archivos} archivos, {total_bytes/1e6:.1f} MB")

# Advertencia (no bloqueante) sobre el limite de 100MB/archivo de GitHub
grandes = [p for p in DIR_RESULTADOS.rglob("*") if p.is_file() and p.stat().st_size > 90_000_000]
if grandes:
    print("\nATENCION -- archivos que se acercan/superan el limite de GitHub (100MB):")
    for p in grandes:
        print(f"  {p.relative_to(RAIZ)}: {p.stat().st_size/1e6:.1f} MB")
else:
    print("\nNingun archivo se acerca al limite de 100MB de GitHub.")


## 2) Estado actual del repositorio Git

Solo lectura: rama, remoto, cambios pendientes.


In [ ]:

run(["git", "remote", "-v"])
run(["git", "branch", "--show-current"])
r = run(["git", "status", "--short"])
cambios = [l for l in r.stdout.splitlines() if l.strip()]
print(f"\n{len(cambios)} rutas con cambios pendientes (nuevas o modificadas).")


## 3) Preparar la publicación (`git add` + `git commit`, local, reversible)

Se agregan únicamente: `RESULTADOS/` (los resultados ya generados, con su
estructura de categorías intacta) y los dos notebooks del pipeline
(`RUSLE_PIPELINE_COMPLETO.ipynb`, este mismo notebook). No se toca
`Entrada/` ni `Salida/` (ambos ignorados por `.gitignore` deliberadamente).


In [ ]:

RUTAS_A_PUBLICAR = [
    str(DIR_RESULTADOS.relative_to(RAIZ)),
    str(NOTEBOOK_PIPELINE.relative_to(RAIZ)) if NOTEBOOK_PIPELINE.exists() else None,
    str(NOTEBOOK_GITHUB.relative_to(RAIZ)) if NOTEBOOK_GITHUB.exists() else None,
]
RUTAS_A_PUBLICAR = [r for r in RUTAS_A_PUBLICAR if r]
print("Rutas que se agregaran al commit:")
for r in RUTAS_A_PUBLICAR:
    print(f"  {r}")

run(["git", "add"] + RUTAS_A_PUBLICAR)
r = run(["git", "status", "--short"])
staged = [l for l in r.stdout.splitlines() if l.startswith(("A ", "M ", "AM"))]
print(f"\n{len(staged)} rutas preparadas (staged) para el commit.")


In [ ]:

FECHA = datetime.now().strftime("%Y-%m-%d %H:%M")
MENSAJE_COMMIT = f"""Resultados RUSLE Amaime: pipeline completo reorganizado ({FECHA})

Publica RESULTADOS/ (factores RUSLE, perdida de suelo, susceptibilidad,
CHIRPS, base de datos procesada, validacion) generado por
RUSLE_PIPELINE_COMPLETO.ipynb. No se modifican datos de entrada.

Co-Authored-By: Claude Sonnet 5 <noreply@anthropic.com>"""

r = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=str(RAIZ))
if r.returncode == 0:
    print("No hay cambios preparados (staged) -- nada que confirmar. Es posible que ya este todo publicado.")
    HAY_CAMBIOS_PARA_COMMIT = False
else:
    run(["git", "commit", "-m", MENSAJE_COMMIT])
    HAY_CAMBIOS_PARA_COMMIT = True
    print("\nCommit creado localmente (todavia NO publicado al remoto).")


## 4) Publicar al remoto (`git push`) — requiere confirmación manual

**Esta celda NO publica nada mientras `CONFIRMAR_PUSH` esté en `False`.**
Revise el resumen del commit arriba; si está correcto, cambie la bandera a
`True` y vuelva a ejecutar esta celda para publicar de verdad en GitHub.


In [ ]:

CONFIRMAR_PUSH = False  # <-- cambiar a True manualmente para publicar de verdad

if not CONFIRMAR_PUSH:
    print("CONFIRMAR_PUSH = False -- no se publico nada. Cambie a True para hacer 'git push'.")
else:
    run(["git", "push", "origin", "HEAD"])
    print("\nPublicado en GitHub.")


## 5) Resumen


In [ ]:

r = run(["git", "log", "-1", "--oneline"])
print("\nUltimo commit local:", r.stdout.strip())
print(f"Resultados publicados desde: {DIR_RESULTADOS}")
print(f"Categorias respetadas: {', '.join(CATEGORIAS_ESPERADAS)}")
print("\nEste notebook no calculo ni modifico ningun dato -- solo publico",
     "lo que RUSLE_PIPELINE_COMPLETO.ipynb ya habia generado.")
